# Silver — Statistics Iceland Monthly Indicators
Spark SQL. Parses the month (`2024M01` → 2024-01-01) and keeps the latest version of each month and metric.

**Source:** `bronze.statistics_iceland_raw` (`dataset IN ('hpi', 'wage_index', 'unemployment_rate')`)  
**Output:** `silver.statistics_monthly_indicators` (Delta), one row per month and metric


In [ ]:
-- Target table (created on the first run)
CREATE TABLE IF NOT EXISTS silver.statistics_monthly_indicators (
    period_raw    STRING,
    metric        STRING,
    year          INT,
    month         INT,
    date          DATE,
    value         DOUBLE,
    processed_at  TIMESTAMP
) USING DELTA

In [ ]:
-- Upsert: the latest ingested version of each (month, metric)
MERGE INTO silver.statistics_monthly_indicators AS target
USING (
    WITH transformed AS (
        SELECT
            period_raw,
            dataset                                         AS metric,
            CAST(value_raw AS DOUBLE)                       AS value,
            TO_DATE(REPLACE(period_raw, 'M', '-'), 'yyyy-MM') AS date,
            ingested_at
        FROM bronze.statistics_iceland_raw
        WHERE dataset IN ('hpi', 'wage_index', 'unemployment_rate')
          AND value_raw IS NOT NULL
    ),
    latest AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY date, metric ORDER BY ingested_at DESC) AS rn
        FROM transformed
        WHERE value IS NOT NULL AND date IS NOT NULL
    )
    SELECT period_raw, metric, YEAR(date) AS year, MONTH(date) AS month, date, value,
           CURRENT_TIMESTAMP() AS processed_at
    FROM latest
    WHERE rn = 1
) AS source
ON target.year = source.year AND target.month = source.month AND target.metric = source.metric
WHEN MATCHED AND target.value <> source.value THEN UPDATE SET
    target.value        = source.value,
    target.date         = source.date,
    target.processed_at = source.processed_at
WHEN NOT MATCHED THEN INSERT (period_raw, metric, year, month, date, value, processed_at)
    VALUES (source.period_raw, source.metric, source.year, source.month, source.date, source.value, source.processed_at)

In [ ]:
-- Check
SELECT metric, COUNT(*) AS months, MIN(date) AS first_month, MAX(date) AS last_month
FROM silver.statistics_monthly_indicators
GROUP BY metric
ORDER BY metric